# Imports

In [14]:
%%capture
# Normally using pip install unsloth is enough

# Temporarily as of Jan 31st 2025, Colab has some issues with Pytorch
# Using pip install unsloth will take 3 minutes, whilst the below takes <1 minute:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post2 peft trl triton
!pip install --no-deps cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer
!pip install --no-deps unsloth

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
BASE_PATH = './'
PROJECT_NAME = 'mesos'

In [3]:
# deterministic sampling 80/20 train/test
import pandas as pd

df_all_tickets = pd.read_csv(BASE_PATH + f'datasets/{PROJECT_NAME}.csv')

train_split_point = int(len(df_all_tickets) * 0.8)

test_tasks = df_all_tickets.iloc[train_split_point:,]

In [4]:
test_tasks = test_tasks.rename(columns={'Custom field (Story Points)': 'storypoint'})
test_tasks

,Unnamed: 0,Summary,storypoint,Assignee_count,Reporter_count,Creator_count
3456,3456,First-order reconciliation of internal service...,14.0,0.073780,0.874542,0.901099
3457,3457,Install supporting services,12.0,0.212259,0.874542,0.901099
3458,3458,Design transfer mechanism for DR tool.,5.0,0.212259,0.874542,0.901099
3459,3459,Monitor Globus transfer tasks,14.0,0.212259,0.874542,0.901099
3460,3460,Process refinement and improvement.,10.0,0.212259,0.874542,0.901099
...,...,...,...,...,...,...
4315,4315,Develop DM requirements for processing crowded...,20.0,0.828604,0.980769,0.980769
4316,4316,Review SpanSet & Footprint API,2.0,0.828604,1.000000,1.000000
4317,4317,Investigate using both monotonicity and symmetry,4.0,0.333712,0.260073,0.260073
4318,4318,Enable fake sources on coadds,6.0,0.334847,1.000000,1.000000


# Inference

In [ ]:
# Create symbolic link
import os


model_name = f"{PROJECT_NAME}_Llama-3.1-8B-Instruct_e10_msl2048_r8"
# for base model, use this instead:
# model_name = f"{PROJECT_NAME}_Llama-3.1-8B-Instruct_base-model"

source_path = os.path.join(BASE_PATH, "models", model_name)
target_path = os.path.join("./", model_name)

!ln -s "{source_path}" "{target_path}"

In [15]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# model_name = "unsloth/Llama-3.1-8B-Instruct"
model_peft, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.9: Fast Llama patching. Transformers: 4.52.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

In [1]:
system_prompt = """You are an expert software development effort estimator. Given a software development issue summary, predict the effort in story points."""
user_prompt = """### Summary:
{}"""
assistant_prompt = """My estimated story point is: {}"""


def generate_convo(row):
  # {"role": "system", "content": "You are an assistant"}
  # {"role": "user", "content": "What is 2+2?"}
  # {"role": "assistant", "content": "It's 4."}

  convo_template = {
    "conversations": [
      {
          "role": "system",
          "content": system_prompt
      },
      {
          "role": "user",
          "content": user_prompt.format(
                          row['Summary'],
                      )
      }
    ]
  }

  return convo_template

import json

# print(json.dumps(generate_convo(test_tasks.iloc[0]), indent=2))

In [17]:
from collections import Counter

def most_frequent(lst):
    return Counter(lst).most_common(1)[0][0]

In [49]:
import re

def get_answer(model, tokenizer, task):
  convo = generate_convo(task)
  messages = convo["conversations"]

  inputs = tokenizer.apply_chat_template(
      messages,
      tokenize = False,
      add_generation_prompt = True, # Must add for generation
      # return_tensors = "pt",
  )
  inputs += "My estimated story point is: "
  inputs = tokenizer(inputs, return_tensors="pt")
  input_ids = inputs['input_ids'].to("cuda")
  attention_mask = inputs['attention_mask'].to("cuda")

  FastLanguageModel.for_inference(model) # Enable native 2x faster inference
  # output = model.generate(input_ids = inputs, max_new_tokens = 5, top_k=1)
  # text = tokenizer.decode(output[0], skip_special_tokens=False)
  outputs = model.generate(
    input_ids = input_ids,
    attention_mask = attention_mask,
    max_new_tokens = 5,
    top_k=1
  )
  text = tokenizer.decode(outputs[0])

  return text

def parse_estimation(answer_text):
    estimation = re.search(r'My estimated story point is:\s*(\d+)', answer_text)
    # estimation = re.search(r'My estimated story point is:\s*(\d+\.?\d*)', answer_text)

    if estimation:
        return estimation.group(1)
    else:
        return 0


In [50]:
import pandas as pd
import random

all_estimations = []
target_dataset = test_tasks
progress_count = 0

print(f"Benchmarking on {len(target_dataset)} questions")

for index, item in target_dataset.iterrows():
    task_id = item['Summary']
    correct_answer = item['storypoint']

    progress_count += 1
    print(f"[{progress_count}/{len(target_dataset)}] - {task_id}")

    llm_output = get_answer(model_peft, tokenizer, item)

    all_estimations.append(
        {
            'task_id': task_id,
            'correct_answer': correct_answer,
            'llm_answer': parse_estimation(llm_output),
            'llm_response': llm_output,
        }
    )


df_results = pd.DataFrame(all_estimations)
df_results['matched'] = df_results['correct_answer'] == df_results['llm_answer']

df_results.head()

Benchmarking on 864 questions
[1/864] - First-order reconciliation of internal service architecture with high-level services
[2/864] - Install supporting services
[3/864] - Design transfer mechanism for DR tool.
[4/864] - Monitor Globus transfer tasks
[5/864] - Process refinement and improvement.
[6/864] - Service Management & Emergent Work  (December) 
[7/864] - Change SQuaSH to store JSON as a BLOB/TEXT in MariaDB 10.1+
[8/864] - DM-wide replanning - December: further development of NCSA WBS
[9/864] - Ops planning - December: assess reviewers feedback, consider descope options 
[10/864] - Commissioning planning - December: contributions to commissioning plan document
[11/864] - Ops planning - January
[12/864] - Commissioning planning - January
[13/864] - DM-wide replanning - January
[14/864] - migrate squash production DB to MariaDB 10.1.x
[15/864] - purge unused jenkins plugins / fully manage all plugins
[16/864] - jenkins master unstable
[17/864] - Remove dependency on CAT package 

,task_id,correct_answer,llm_answer,llm_response,matched
0,First-order reconciliation of internal service...,14.0,8,<|begin_of_text|><|begin_of_text|><|start_head...,False
1,Install supporting services,12.0,8,<|begin_of_text|><|begin_of_text|><|start_head...,False
2,Design transfer mechanism for DR tool.,5.0,8,<|begin_of_text|><|begin_of_text|><|start_head...,False
3,Monitor Globus transfer tasks,14.0,8,<|begin_of_text|><|begin_of_text|><|start_head...,False
4,Process refinement and improvement.,10.0,5,<|begin_of_text|><|begin_of_text|><|start_head...,False


In [55]:
df_results.to_csv(BASE_PATH + f"results/{model_name.replace('/', '-')}.csv", index=False)
print(BASE_PATH + f"results/{model_name.replace('/', '-')}.csv")

In [53]:
df_results['llm_answer'] = pd.to_numeric(df_results['llm_answer'], errors='coerce')
df_results['correct_answer'] = pd.to_numeric(df_results['correct_answer'], errors='coerce')

In [54]:
import numpy as np

def calculate_mae(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    return np.mean(np.abs(actual - predicted))

def calculate_mmre(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)

    relative_errors = np.abs(actual - predicted) / actual
    mmre = np.mean(relative_errors)
    return mmre

def calculate_pred(actual, predicted, threshold=0.50):
    actual = np.array(actual)
    predicted = np.array(predicted)

    relative_errors = np.abs(actual - predicted) / actual
    pred = np.mean(relative_errors <= threshold)
    return pred

actual_values = df_results['correct_answer'].tolist()
predicted_values = df_results['llm_answer'].tolist()
mae = calculate_mae(actual_values, predicted_values)
mmre = calculate_mmre(actual_values, predicted_values)
pred = calculate_pred(actual_values, predicted_values, 0.5)
print(f"Mean Absolute Error (MAE): {mae}")
print(f"Mean Magnitude of Relative Error (MMRE): {mmre}")
print(f"Prediction within 50% of actual value (PRED(50)): {pred}")

Mean Absolute Error (MAE): 17.053712962962962
Mean Magnitude of Relative Error (MMRE): 11.698226047820548
Prediction within 50% of actual value (PRED(50)): 0.1574074074074074
